# Прогнозирование мощности ветрогенератора

## Сравнение моделей

В работе сравниваются несколько подходов к прогнозированию
нормализованной активной мощности:

- CatBoost
- SimpleRNN
- LSTM
- SARIMA

Для сравнения используются одинаковые тестовые данные и метрики:

- MAE — средняя абсолютная ошибка;
- RMSE — корень из среднеквадратичной ошибки.

Чем меньше значения MAE и RMSE, тем меньше ошибка прогноза.

In [52]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.style.use("seaborn-v0_8-whitegrid")

In [53]:
ARTIFACTS_DIR = "model_artifacts"

print(os.listdir(ARTIFACTS_DIR))


['catboost_predictions.csv', 'catboost_wind_power.cbm', 'lstm_predictions.csv', 'lstm_wind_power.keras', 'rnn_predictions.csv', 'rnn_wind_power.keras']


In [54]:
rnn_results = pd.read_csv(
    os.path.join(
        ARTIFACTS_DIR,
        "rnn_predictions.csv"
    )
)

lstm_results = pd.read_csv(
    os.path.join(
        ARTIFACTS_DIR,
        "lstm_predictions.csv"
    )
)

catboost_results = pd.read_csv(
    os.path.join(
        ARTIFACTS_DIR,
        "catboost_predictions.csv"
    )
)

print("RNN:")
display(rnn_results.head())

print("\nLSTM:")
display(lstm_results.head())

print("\nCATBOOST:")
display(catboost_results.head())

RNN:


,actual,prediction,error
0,0.44,0.418623,0.021377
1,0.40,0.426235,-0.026235
2,0.40,0.374243,0.025757
3,0.39,0.380154,0.009846
4,0.22,0.362091,-0.142091



LSTM:


,actual,prediction,error
0,0.44,0.404097,0.035903
1,0.40,0.407759,-0.007759
2,0.40,0.373869,0.026131
3,0.39,0.375026,0.014973
4,0.22,0.358443,-0.138443



CATBOOST:


,datetime,forecast_normalized_power,forecast_energy_mwh,forecast_power_mw
0,2026-09-23 15:00:00,0.013867,0.693363,0.693363
1,2026-09-23 16:00:00,0.042504,2.125213,2.125213
2,2026-09-23 17:00:00,0.095940,4.797017,4.797017
3,2026-09-23 18:00:00,0.150934,7.546692,7.546692
4,2026-09-23 19:00:00,0.134272,6.713591,6.713591


In [55]:
def calculate_metrics(y_true, y_pred):
    
    mae = mean_absolute_error(
        y_true,
        y_pred
    )
    
    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )
    
    return mae, rmse


In [56]:
rnn_mae, rnn_rmse = calculate_metrics(
    rnn_results["actual"],
    rnn_results["prediction"]
)

print(f"RNN MAE  : {rnn_mae:.6f}")
print(f"RNN RMSE : {rnn_rmse:.6f}")


RNN MAE  : 0.046756
RNN RMSE : 0.077491


In [57]:
lstm_mae, lstm_rmse = calculate_metrics(
    lstm_results["actual"],
    lstm_results["prediction"]
)

print(f"LSTM MAE  : {lstm_mae:.6f}")
print(f"LSTM RMSE : {lstm_rmse:.6f}")


LSTM MAE  : 0.047301
LSTM RMSE : 0.077670


## SARIMA

SARIMA также была рассмотрена в качестве модели прогнозирования
временного ряда.

Для данного набора данных SARIMA является обоснованным подходом,
поскольку целевая переменная представляет собой временной ряд с
регулярным 10-минутным интервалом и потенциальной суточной
сезонностью.

В частности, при интервале 10 минут в одних сутках находится:

144 = 24 × 60 / 10

наблюдения.

Поэтому естественным вариантом является сезонный период s = 144.

Однако обучение SARIMA на полном наборе данных оказалось
вычислительно значительно более длительным по сравнению с
нейросетевыми моделями и CatBoost. Из-за этого полноценный
эксперимент с SARIMA на всём наборе данных не был завершён в
отведённое время.

Поэтому SARIMA не включается в числовой рейтинг моделей: для неё
отсутствуют результаты прогноза на том же тестовом наборе.

При этом это не означает, что SARIMA является плохой моделью.
Напротив, она является классическим и вполне подходящим методом
для временных рядов и особенно интересна в качестве baseline
для сравнения с машинным обучением и нейронными сетями.

Для окончательного ранжирования SARIMA необходимо завершить её
обучение и получить прогноз на том же тестовом периоде.


# Итоговое сравнение

В работе были рассмотрены различные подходы к прогнозированию
нормализованной активной мощности ветрогенератора.

Для оценки использовались MAE и RMSE на отложенной тестовой
выборке. Чем меньше значение метрики, тем меньше ошибка прогноза.

CatBoost использует преимущества градиентного бустинга и хорошо
подходит для табличных данных и нелинейных зависимостей между
скоростью ветра, температурой и мощностью.

RNN способна учитывать последовательную структуру временного ряда,
однако простая рекуррентная архитектура может хуже сохранять
информацию о более длинных временных зависимостях.

LSTM является развитием RNN и использует механизм памяти,
позволяющий лучше учитывать долгосрочные зависимости во временном
ряду.

SARIMA представляет классический статистический подход. Для
данного временного ряда она является обоснованным вариантом,
особенно с учётом регулярного 10-минутного интервала и возможной
суточной сезонности. Однако обучение модели на полном объёме данных
оказалось значительно более длительным, поэтому её итоговые
метрики на том же тестовом наборе в текущем эксперименте получить
не удалось.

Таким образом, количественный вывод о моделях делается на основе
фактически рассчитанных MAE и RMSE, а SARIMA рассматривается как
отдельный классический baseline, требующий завершения расчёта для
полного сравнения.
